In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis
from scite.cs_design.shapes import RectangularShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.cs_design.cs_stress_strain_profile import StressStrainProfile
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import create_steel

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print('Imports OK')

## Cross-section definition (same as mb3 notebook)

Symmetric section: $A_{s,bot} = A_{s,top} = 1257\,\text{mm}^2$, C30/37, B500B.

In [ ]:
b_ref, h_ref = 300.0, 700.0
A_s_top_ref  = 1610.0 # 1257.0
A_s_bot_ref  = 1257.0
z_s_ref      = 50.0
z_s_top      = h_ref - z_s_ref
L_ref        = 6000.0

concrete_ref = EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5)
steel        = create_steel('B500B')

shape_rect = RectangularShape(b=b_ref, h=h_ref)
layer_bot  = ReinforcementLayer(material=steel, A_s=A_s_bot_ref, z=z_s_ref)
layer_top  = ReinforcementLayer(material=steel, A_s=A_s_top_ref, z=z_s_top)
reinf_ref  = ReinforcementLayout(layers=[layer_bot, layer_top])
cs_rect    = CrossSection(shape=shape_rect, concrete=concrete_ref,
                          reinforcement=reinf_ref)

bda = BeamDeflectionAnalysis(
    cs=cs_rect, L=L_ref, load_type='dist',
    n_x=200, n_kappa=150, n_load_steps=41,
    solve_mode='both',
)

print(f'Section: {b_ref:.0f}×{h_ref:.0f} mm')
print(f'  A_s,bot = {A_s_bot_ref:.0f} mm²  at z = {z_s_ref:.0f} mm')
print(f'  A_s,top = {A_s_top_ref:.0f} mm²  at z = {z_s_top:.0f} mm')
print(f'M_R = {bda.M_R:.3f} kN·m')
print(f'Symmetric section: A_s,bot == A_s,top → {A_s_bot_ref == A_s_top_ref}')

## Constitutive laws

In [ ]:
fig, (ax_c, ax_s) = plt.subplots(1, 2, figsize=(12, 4.5))

concrete_ref.plot_stress_strain(ax_c)
ax_c.set_title(
    f'Concrete  C{concrete_ref.f_ck:.0f}/37  '
    f'($f_{{cd}}$ = {concrete_ref.f_cd:.2f} MPa,  '
    f'$\\varepsilon_{{cu2}}$ = {abs(concrete_ref.eps_cu2_computed)*1e3:.1f} ‰)',
    fontsize=10, fontweight='bold'
)

steel.plot_stress_strain(ax_s)
ax_s.set_title(
    f'Reinforcement  B500B  '
    f'($f_{{yd}}$ = {steel.f_yd:.1f} MPa,  '
    f'$f_{{td}}$ = {steel.f_td:.1f} MPa,  '
    f'$\\varepsilon_{{ud}}$ = {steel.eps_ud*1e3:.1f} ‰)',
    fontsize=10, fontweight='bold'
)

plt.tight_layout()
plt.show()

## M-κ curve — full range (sagging + hogging)

For a **symmetric** section both branches must be mirror images. Any difference
in the kappa extent (positive vs negative side) reveals a bug in the solver.

In [ ]:
mk = bda.mk

fig, ax = plt.subplots(figsize=(10, 5))
mk.plot_mk(ax=ax, show_full_range=True, show_failure_marker=True)
ax.set_title('M-κ curve — symmetric section (should be anti-symmetric)', fontweight='bold')
plt.tight_layout()
plt.show()

# kappa_neg_values goes from small-magnitude (near 0) to large-magnitude (most negative)
kappa_max_pos = mk.kappa_values[-1]   * 1e3   # ‰/mm  (largest positive)
kappa_max_neg = mk.kappa_neg_values[-1] * 1e3  # ‰/mm  (most negative = largest magnitude)
print(f'Positive branch: κ ∈ [{mk.kappa_values[0]*1e3:.5f}, {kappa_max_pos:.5f}] ‰/mm  '
      f'n={len(mk.kappa_values)}')
print(f'Negative branch: κ ∈ [{mk.kappa_neg_values[0]*1e3:.5f}, {kappa_max_neg:.5f}] ‰/mm  '
      f'n={len(mk.kappa_neg_values)}')
print(f'|κ_max_pos| = {abs(kappa_max_pos):.5f} ‰/mm')
print(f'|κ_max_neg| = {abs(kappa_max_neg):.5f} ‰/mm')
print(f'Asymmetry = {abs(kappa_max_neg)/abs(kappa_max_pos):.4f}  (should be 1.0000 for symmetric section)')

## State at peak positive moment (sagging)

Uses `mk.plot_state_at_kappa(kappa_R_sag)` — finds closest point in the
positive kappa array and shows strain + stress profiles.

In [ ]:
idx_sag = int(np.argmax(mk.M_values))
kappa_R_sag  = mk.kappa_values[idx_sag]
M_R_sag      = mk.M_values[idx_sag]
eps_bot_sag  = mk.eps_bot_values[idx_sag]
eps_top_sag  = eps_bot_sag - kappa_R_sag * h_ref

print(f'Sagging peak:  κ = {kappa_R_sag*1e3:.5f} ‰/mm   M = {M_R_sag:.3f} kN·m')
print(f'  ε_bot = {eps_bot_sag*1e3:.3f} ‰   ε_top = {eps_top_sag*1e3:.3f} ‰')

fig, (ax_strain, ax_stress) = plt.subplots(1, 2, figsize=(8, 6))
fig.suptitle(
    f'Sagging peak  κ = {kappa_R_sag*1e3:.4f} ‰/mm   M = {M_R_sag:.2f} kN·m',
    fontsize=13, fontweight='bold'
)
mk.plot_state_at_kappa(kappa_R_sag, ax_strain=ax_strain, ax_stress=ax_stress)
plt.tight_layout()
plt.show()

## State at peak negative moment (hogging)

The hogging branch is stored in `mk.kappa_neg_values` / `mk.eps_bot_neg_values`.
We build a `StressStrainProfile` directly and call its plot method.

In [ ]:
idx_hog = int(np.argmin(mk.M_neg_values))   # most negative M
kappa_R_hog  = mk.kappa_neg_values[idx_hog]
M_R_hog      = mk.M_neg_values[idx_hog]
eps_bot_hog  = mk.eps_bot_neg_values[idx_hog]
eps_top_hog  = eps_bot_hog - kappa_R_hog * h_ref

print(f'Hogging peak:  κ = {kappa_R_hog*1e3:.5f} ‰/mm   M = {M_R_hog:.3f} kN·m')
print(f'  ε_bot = {eps_bot_hog*1e3:.3f} ‰   ε_top = {eps_top_hog*1e3:.3f} ‰')

profile_hog = StressStrainProfile(cs_rect, kappa=kappa_R_hog, eps_bottom=eps_bot_hog)

fig, (ax_strain, ax_stress) = plt.subplots(1, 2, figsize=(8, 6))
fig.suptitle(
    f'Hogging peak  κ = {kappa_R_hog*1e3:.4f} ‰/mm   M = {M_R_hog:.2f} kN·m',
    fontsize=13, fontweight='bold'
)
profile_hog.plot_stress_strain_profile(ax_strain, ax_stress, show_resultants=True)
plt.tight_layout()
plt.show()

## Symmetry check summary

For a perfectly symmetric section the following pairs must be equal in magnitude.

In [ ]:
print('=== M-κ symmetry check ===')
print(f'  |M_R_sag|  = {abs(M_R_sag):.4f} kN·m')
print(f'  |M_R_hog|  = {abs(M_R_hog):.4f} kN·m')
print(f'  ratio      = {abs(M_R_hog)/abs(M_R_sag):.6f}  (should be 1.0)')
print()
print(f'  |κ_R_sag|  = {abs(kappa_R_sag)*1e3:.5f} ‰/mm')
print(f'  |κ_R_hog|  = {abs(kappa_R_hog)*1e3:.5f} ‰/mm')
print(f'  ratio      = {abs(kappa_R_hog)/abs(kappa_R_sag):.6f}  (should be 1.0)')
print()
eps_cu = abs(concrete_ref.eps_cu2_computed)
eps_ud = steel.eps_ud
kappa_max_design = (eps_cu + eps_ud) / h_ref * 1e3  # ‰/mm
print(f'  Designed kappa_max = {kappa_max_design:.5f} ‰/mm  (both branches, balanced failure)')
print(f'  Positive branch reached {abs(kappa_max_pos)/kappa_max_design*100:.1f}% of designed kappa_max')
print(f'  Negative branch reached {abs(kappa_max_neg)/kappa_max_design*100:.1f}% of designed kappa_max')

---
## Equal-span continuous beam — M_hog(q) analysis

Two equal spans, same symmetric cross-section. The κ-controlled solver sweeps
the load from zero to the ultimate state where $M_\mathrm{hog,nl} \to M_R$.

In [ ]:
from scite.beam.continuous import ContinuousBeamAnalysis, CompatibilityNoRoot

La, Lb = bda.L, bda.L

print(f'M_R = {bda.M_R:.3f} kN·m')

# ── κ_hog-controlled sweep ────────────────────────────────────────────────────
# Outer parameter: κ_hog steps from 0 to κ_R,hog taken directly from the M-κ table.
# The compatible distributed load q is solved at each step from rotation compatibility.
# This always reaches full section capacity — no convergence truncation.
cba_d = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=1.0)
s = cba_d.sweep
kappa_hog  = s['kappa_hog']   # [1/mm]  signed hogging curvatures
M_nl_hog   = s['M_nl_hog']    # [kN·m]  nonlinear hogging moments
q          = s['q']           # [N/mm]  compatible loads
M_el_hog   = s['M_el_hog']    # [kN·m]  elastic hogging at each step
M_nl_sag_a = s['M_nl_sag_a']  # [kN·m]  max NL sagging in left span
M_el_sag_a = s['M_el_sag_a']  # [kN·m]  max elastic sagging in left span

q_u = q[-1]
# elastic load equivalent to M_nl_hog at each step (for redistribution ratio)
q_el_hog = -M_el_hog * 8.0 * (La + Lb) / (La**3 + Lb**3) * 1e6  # [N/mm]

print(f'\nSweep: {len(kappa_hog)} steps')
print(f'  q_hog_u   = {cba_d.q_hog_u:.4f} N/mm  ({cba_d.q_hog_u:.2f} kN/m)')
print(f'  |M_R,hog| = {abs(M_nl_hog[-1]):.3f} kN·m  at |κ| = {abs(kappa_hog[-1])*1e3:.5f} ‰/mm')

# ── Redistribution: M(q) diagram ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(q, np.abs(M_el_hog), '--', color='#666666', lw=1.5,
        label='$|M_{hog}|$ elastic')
ax.plot(q, np.abs(M_nl_hog),    '-',  color='black',   lw=2.0,
        label='$|M_{hog}|$ nonlinear')
ax.plot(q, M_el_sag_a, '--', color='steelblue', lw=1.5,
        label='$M_{sag}$ elastic')
ax.plot(q, M_nl_sag_a,    '-',  color='#005599',   lw=2.0,
        label='$M_{sag}$ nonlinear')
ax.axhline(bda.M_R, color='#888888', ls=':', lw=1.0,
           label=f'$M_R$ = {bda.M_R:.1f} kN·m')
ax.axvline(q_u, color='#2ca02c', ls='--', lw=1.2,
           label=f'$q_{{hog,u}}$ = {q_u:.2f} N/mm')
ax.set_xlabel('$q$  [N/mm = kN/m]', fontsize=11)
ax.set_ylabel('Moment  [kN·m]', fontsize=11)
ax.set_title('Redistribution: hogging and sagging moments vs. load',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

δ_u = float(q_el_hog[-1]) / q_u
print(f'\nAt ultimate:  δ_u = q_el/q_u = {δ_u:.4f}  →  redistribution {(1-δ_u)*100:.1f} %')
print(f'M_sag at q_u = {M_nl_sag_a[-1]:.3f} kN·m  (section reserve: {(1 - M_nl_sag_a[-1]/bda.M_R)*100:.1f} %)')

## Bending moment profile at elastic reference load $q_\mathrm{field,u}$

At $q_\mathrm{field,u} = 8 M_R / L^2$ an elastic analysis gives $M_\mathrm{hog,el} = -M_R$
at the interior support — this is the point at which the elastic beam "fails".
The Strategy D sweep shows the actual ultimate load $q_u > q_\mathrm{field,u}$,
enabled by moment redistribution from the interior support to the sagging zone.

In [ ]:
# ── Elastic moment profile at q = 8·M_R/L² = bda.F_R ─────────────────────────
# At this reference load the elastic analysis gives M_el_hog = -M_R exactly.
La = bda.L
cba_el = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=float(bda.F_R))
M_el_hog_ref_Nmm = cba_el.M_el_hog   # N·mm, = -bda.M_R * 1e6
x_a, M_a, _, _   = cba_el.M_profiles(M_el_hog_ref_Nmm)

idx_sag = int(np.argmax(M_a))
x_sag   = x_a[idx_sag]
M_sag   = M_a[idx_sag]   # N·mm

print(f'Elastic reference at q = {bda.F_R:.4f} N/mm  (= 8·M_R/L²):')
print(f'  M_el,hog = {M_el_hog_ref_Nmm/1e6:.3f} kN·m  (= −M_R = −{bda.M_R:.3f} kN·m)')
print(f'  M_sag,el = {M_sag/1e6:.3f} kN·m  at x = {x_sag/1e3:.3f} m')
print(f'  Ratio M_sag/|M_hog| = {M_sag/abs(M_el_hog_ref_Nmm):.4f}  (9/16 = {9/16:.4f} elastic two-span)')

fig_m, ax_m = plt.subplots(figsize=(10, 3.5))
ax_m.plot(x_a / 1e3, M_a / 1e6, '-', color='steelblue', lw=2,
          label='$M(x)$ elastic at $q = 8M_R/L^2$')
ax_m.axhline(0, color='k', lw=0.5)
ax_m.scatter([x_sag / 1e3], [M_sag / 1e6], color='orange', zorder=5, s=60,
             label=f'Max sagging  x = {x_sag/1e3:.2f} m   M = {M_sag/1e6:.2f} kN·m')
ax_m.scatter([La / 1e3], [M_el_hog_ref_Nmm / 1e6], color='#9467bd', zorder=5, s=60,
             label=f'Interior support  M = {M_el_hog_ref_Nmm/1e6:.2f} kN·m = $-M_R$')
ax_m.set_xlabel('$x$  [m]', fontsize=11)
ax_m.set_ylabel('$M$  [kN·m]', fontsize=11)
ax_m.set_title(
    f'Elastic moment profile at $q = 8M_R/L^2$ = {bda.F_R:.2f} N/mm'
    f'  —  elastic "failure": hogging reaches $M_R$',
    fontweight='bold'
)
ax_m.legend(fontsize=9)
ax_m.grid(True, alpha=0.3, ls='--')
ax_m.invert_yaxis()   # sagging below, hogging above
plt.tight_layout()
plt.show()

---
## κ_hog-controlled sweep

$\kappa_{hog} \leq 0$ steps through the ascending hogging M-κ branch taken directly
from the M-κ table, and the compatible distributed load $q$ is solved from rotation
compatibility at each step.

**Why κ-controlled and not a q-sweep?**  A load sweep may fail to converge before
$M_{hog}$ reaches $M_R$ (the root-finding bracket collapses as the nonlinear response
stiffens/softens), making it impossible to reliably determine $q_u$.  By prescribing
$\kappa_{hog}$ from the pre-computed M-κ table and solving for $q$, every step is
guaranteed to converge and the last step gives $q_u$ directly.

**Variables extracted from `cba_d.sweep` in cell-14:**
$\kappa_{hog},\ M_{nl,hog},\ q,\ M_{el,hog},\ M_{nl,sag,a},\ M_{el,sag,a}$

In [ ]:
# ── x/d hogging section along the κ_hog sweep ────────────────────────────────
eps_bot_sweep = mk.eps_bot_neg_values[:len(kappa_hog)]
xd_hog_arr   = np.empty(len(kappa_hog))
mode_arr      = []
for i, (k, eps_b) in enumerate(zip(kappa_hog, eps_bot_sweep)):
    p = StressStrainProfile(cs_rect, kappa=float(k), eps_bottom=float(eps_b))
    _, _, xd_hog_arr[i] = p.xd
    mode_arr.append(p.failure_mode)

# ── x/d sagging section — M_nl_sag_a(κ_hog) → κ_sag via M-κ table → x/d ────
M_pos_ext  = np.concatenate([[0.0], mk.M_values])
k_pos_ext  = np.concatenate([[0.0], mk.kappa_values])
eb_pos_ext = np.concatenate([[0.0], mk.eps_bot_values])
xd_sag_arr = np.full(len(kappa_hog), np.nan)
for i, M_sag_i in enumerate(M_nl_sag_a):
    k_sag = float(np.interp(M_sag_i, M_pos_ext, k_pos_ext))
    if k_sag < 1e-9:
        continue
    eb_sag = float(np.interp(k_sag, k_pos_ext, eb_pos_ext))
    p_sag  = StressStrainProfile(cs_rect, kappa=k_sag, eps_bottom=eb_sag)
    _, _, xd_sag_arr[i] = p_sag.xd

# ── yield and failure bookmarks ───────────────────────────────────────────────
eps_top_sweep = eps_bot_sweep - kappa_hog * h_ref
eps_yd_val    = steel.eps_yd
yield_mask    = eps_top_sweep >= eps_yd_val
i_yield       = int(np.argmax(yield_mask)) if yield_mask.any() else len(kappa_hog)
final_mode    = mode_arr[-1]
kappa_abs_permm = np.abs(kappa_hog) * 1e3   # ‰/mm

C_HOG  = 'black'
C_SAG  = '#005599'
C_YLD  = '#2ca02c'
C_FAIL = '#d62728'

kv = kappa_abs_permm[i_yield] if yield_mask.any() else None
ku = kappa_abs_permm[-1]

# Panel 2 reference values
M_el_hog_at_qu = abs(M_el_hog[-1])       # kN·m — elastic M_hog at q_u (> M_R,hog)
M_R_hog_val    = abs(M_nl_hog[-1])       # kN·m — hogging section capacity
delta_M        = cba_d.delta_M_u         # redistribution factor

# ── Triple-panel figure — shared x-axis, no vertical gap ─────────────────────
fig, (ax_q, ax_m, ax_xd) = plt.subplots(
    3, 1, figsize=(9, 10), sharex=True,
    gridspec_kw={'height_ratios': [2, 2, 1]}
)
plt.subplots_adjust(hspace=0, top=0.94, bottom=0.07, left=0.13, right=0.97)

ax_q.tick_params(labelbottom=False, bottom=False)
ax_m.tick_params(labelbottom=False, bottom=False)

def _vlines(ax):
    if kv is not None:
        ax.axvline(kv, color=C_YLD,  ls='--', lw=1.1, zorder=0)
    ax.axvline(ku, color=C_FAIL, ls=':',  lw=1.1, zorder=0)

# ─── Panel 1: q(|κ_hog|) ──────────────────────────────────────────────────────
ax_q.set_title('κ_hog-controlled sweep',
               fontsize=11, fontweight='bold', pad=6)
ax_q.plot(kappa_abs_permm[:i_yield + 1], q[:i_yield + 1], '-',
          color='steelblue', lw=2.0, label='$q$  pre-yield')
if i_yield < len(kappa_hog) - 1:
    ax_q.plot(kappa_abs_permm[i_yield:], q[i_yield:], '-',
              color='#9467bd', lw=2.0, label='$q$  post-yield')
ax_q.axhline(q_u, color=C_FAIL, ls=':', lw=1.0,
             label=f'$q_{{hog,u}}$ = {q_u:.2f} N/mm')
_vlines(ax_q)
ax_q.set_ylabel('$q$  [N/mm]', fontsize=11)
ax_q.legend(fontsize=11, loc='lower right')
ax_q.grid(True, alpha=0.3, ls='--')

if kv is not None:
    ax_q.annotate(
        f'top-bar yield\n$\\varepsilon_{{yd}}={eps_yd_val*1e3:.2f}$‰',
        xy=(kv, q[i_yield]), xytext=(kv, 0.70 * q_u),
        arrowprops=dict(arrowstyle='->', color=C_YLD, lw=1.2),
        color=C_YLD, fontsize=8.5, ha='left', va='top'
    )
ax_q.annotate(
    f'$q_{{hog,u}}$={q_u:.2f}\n{final_mode}',
    xy=(ku, q_u), xytext=(ku, 0.70 * q_u),
    arrowprops=dict(arrowstyle='->', color=C_FAIL, lw=1.2),
    color=C_FAIL, fontsize=8.5, ha='right', va='top'
)

# ─── Panel 2: |M_hog|, M_sag, M_R references, elastic M_hog at q_u ───────────
ax_m.plot(kappa_abs_permm, np.abs(M_nl_hog), '-', color=C_HOG, lw=2.0,
          label='$|M_{nl,hog}|$')
ax_m.plot(kappa_abs_permm, M_nl_sag_a, '-', color=C_SAG, lw=2.0,
          label='$M_{nl,sag}$')
ax_m.axhline(bda.M_R,        color=C_SAG, ls=':', lw=1.2,
             label=f'$M_R$ (sag) = {bda.M_R:.1f} kN·m')
ax_m.axhline(M_R_hog_val,    color=C_HOG, ls=':', lw=1.2,
             label=f'$M_R$ (hog) = {M_R_hog_val:.1f} kN·m')
ax_m.axhline(M_el_hog_at_qu, color='#666666', ls='--', lw=1.2,
             label=f'$|M_{{el,hog}}|(q_{{hog,u}})$ = {M_el_hog_at_qu:.1f} kN·m'
                   f'   ($\\delta_M = {delta_M:.3f}$)')
_vlines(ax_m)
ax_m.set_ylim(bottom=0, top=M_el_hog_at_qu * 1.12)
ax_m.set_ylabel('Moment  [kN·m]', fontsize=11)
ax_m.legend(fontsize=11, loc='lower right')
ax_m.grid(True, alpha=0.3, ls='--')

# ─── Panel 3: x/d ─────────────────────────────────────────────────────────────
ax_xd.plot(kappa_abs_permm, xd_hog_arr, '-', color=C_HOG, lw=2.0,
           label='$x/d$  hogging')
ax_xd.plot(kappa_abs_permm, xd_sag_arr, '-', color=C_SAG, lw=2.0,
           label='$x/d$  sagging')
ax_xd.axhline(0.45, color='gray', ls=':', lw=1.0, label='EC2  $x/d = 0.45$')
_vlines(ax_xd)
ax_xd.set_xlabel('$|\\kappa_{hog}|$  [‰/mm]', fontsize=11)
ax_xd.set_ylabel('$x/d$', fontsize=11)
ax_xd.legend(fontsize=11, loc='upper right')
ax_xd.grid(True, alpha=0.3, ls='--')

fig.align_ylabels([ax_q, ax_m, ax_xd])
plt.show()

# ── Summary ────────────────────────────────────────────────────────────────────
δ_u = float(q_el_hog[-1]) / q_u
ec2_min = max(0.64 + 0.8 * xd_hog_arr[-1], 0.80)
ec2_ok  = δ_u >= ec2_min
print(f'At κ_R,hog:')
print(f'  Failure mode     : {final_mode}')
print(f'  x/d (hog)        : {xd_hog_arr[-1]:.4f}  (EC2 ductility limit: 0.45)')
if not np.isnan(xd_sag_arr[-1]):
    print(f'  x/d (sag)        : {xd_sag_arr[-1]:.4f}')
if yield_mask.any():
    print(f'  Top-bar yield    : |κ| = {kappa_abs_permm[i_yield]:.5f} ‰/mm'
          f',  q = {q[i_yield]:.2f} N/mm  (step {i_yield}/{len(kappa_hog)-1})')
print(f'  M_R (hog)        = {M_R_hog_val:.2f} kN·m')
print(f'  M_R (sag)        = {bda.M_R:.2f} kN·m')
print(f'  M_el,hog(q_hog_u) = {M_el_hog_at_qu:.2f} kN·m')
print(f'  δ_M = |M_nl,hog_u| / |M_el,hog(q_hog_u)| = {delta_M:.4f}'
      f'  →  moment redistribution {(1-delta_M)*100:.1f}%')
print(f'  δ_u (load-based) = {δ_u:.4f}  →  load redistribution {(1-δ_u)*100:.1f} %')
print(f'  EC2 check: δ={δ_u:.4f} {"≥" if ec2_ok else "<"} '
      f'{ec2_min:.4f}  → {"OK" if ec2_ok else "FAIL"}')

In [ ]:
# ── Cross-section states at q_hog_u (hogging-governed ultimate state) ────────
# κ_hog = κ_R,hog (last step of the sweep = hogging section at full capacity)
kappa_hog_u  = kappa_hog[-1]          # signed [1/mm] ≤ 0
M_nl_hog_u   = M_nl_hog[-1] * 1e6    # signed [N·mm] ≤ 0

cba_ult = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=float(q_u))
x_a_u, M_a_u, _, _ = cba_ult.M_profiles(M_nl_hog_u)

idx_sag_u   = int(np.argmax(M_a_u))
x_sag_u     = x_a_u[idx_sag_u]
M_sag_u     = M_a_u[idx_sag_u]      # N·mm, positive (sagging)

La_u = bda.L   # interior support x-coordinate

print(f'Ultimate state:  q_hog_u = {q_u:.4f} N/mm')
print(f'  Hogging section:  κ = {kappa_hog_u*1e3:.5f} ‰/mm   M = {M_nl_hog_u/1e6:.3f} kN·m  (signed)')
print(f'  Max sagging:      M = {M_sag_u/1e6:.3f} kN·m   at x = {x_sag_u/1e3:.3f} m')

# ── Interpolate ε_bot for each section from M-κ table ────────────────────────
eps_bot_hog_u = float(np.interp(
    abs(kappa_hog_u),
    -mk.kappa_neg_values,        # magnitudes, ascending
    mk.eps_bot_neg_values
))

# Sagging — M → κ → ε_bot from positive branch
kappa_sag_u   = float(np.interp(M_sag_u / 1e6, mk.M_values, mk.kappa_values))
eps_bot_sag_u = float(np.interp(kappa_sag_u, mk.kappa_values, mk.eps_bot_values))

print(f'\n  Hogging section:  κ = {kappa_hog_u*1e3:.5f} ‰/mm   ε_bot = {eps_bot_hog_u*1e3:.3f} ‰')
print(f'  Sagging section:  κ = {kappa_sag_u*1e3:.5f} ‰/mm   ε_bot = {eps_bot_sag_u*1e3:.3f} ‰')

# ── Build StressStrainProfile objects ─────────────────────────────────────────
profile_hog_u = StressStrainProfile(cs_rect, kappa=kappa_hog_u, eps_bottom=eps_bot_hog_u)
profile_sag_u = StressStrainProfile(cs_rect, kappa=kappa_sag_u, eps_bottom=eps_bot_sag_u)

# ── Moment diagram at q_hog_u ─────────────────────────────────────────────────
fig_m, ax_m = plt.subplots(figsize=(10, 3.5))
ax_m.plot(x_a_u / 1e3, M_a_u / 1e6, '-', color='#9467bd', lw=2.0, label='$M(x)$ span left')
ax_m.axhline(0, color='k', lw=0.5)
ax_m.scatter([x_sag_u / 1e3], [M_sag_u / 1e6], color='orange',  zorder=5, s=70,
             label=f'Max sagging  x = {x_sag_u/1e3:.2f} m   M = {M_sag_u/1e6:.2f} kN·m')
ax_m.scatter([La_u / 1e3], [M_nl_hog_u / 1e6], color='#2ca02c', zorder=5, s=70,
             label=f'Interior support  x = {La_u/1e3:.1f} m   M = {M_nl_hog_u/1e6:.2f} kN·m = $M_R$')
ax_m.set_xlabel('$x$  [m]', fontsize=11)
ax_m.set_ylabel('$M$  [kN·m]  ($+$ sagging $\\downarrow$, $-$ hogging $\\uparrow$)', fontsize=11)
ax_m.set_title(f'Bending moment at $q_{{hog,u}}$ = {q_u:.2f} N/mm', fontweight='bold')
ax_m.legend(fontsize=9)
ax_m.grid(True, alpha=0.3, ls='--')
ax_m.invert_yaxis()
plt.tight_layout()
plt.show()

# ── 2×2 cross-section panel ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(
    f'Cross-section states at $q_{{hog,u}}$ = {q_u:.2f} N/mm'
    f'  ($\\kappa_{{hog}} = \\kappa_{{R,hog}}$)',
    fontsize=13, fontweight='bold'
)

ax_hs, ax_hst = axes[0]   # hogging: strain | stress
ax_ss, ax_sst = axes[1]   # sagging: strain | stress

profile_hog_u.plot_stress_strain_profile(ax_hs, ax_hst, show_resultants=True)
ax_hs.set_title(
    f'HOGGING — interior support  ($\\kappa = \\kappa_{{R,hog}}$)\n'
    f'κ = {kappa_hog_u*1e3:.4f} ‰/mm   M = {M_nl_hog_u/1e6:.2f} kN·m',
    fontsize=10, fontweight='bold', color='#2ca02c'
)
ax_hst.set_title('HOGGING — stress', fontsize=10, fontweight='bold', color='#2ca02c')

profile_sag_u.plot_stress_strain_profile(ax_ss, ax_sst, show_resultants=True)
ax_ss.set_title(
    f'SAGGING — max positive moment at $x$ = {x_sag_u/1e3:.2f} m\n'
    f'κ = {kappa_sag_u*1e3:.4f} ‰/mm   M = {M_sag_u/1e6:.2f} kN·m',
    fontsize=10, fontweight='bold', color='#9467bd'
)
ax_sst.set_title('SAGGING — stress', fontsize=10, fontweight='bold', color='#9467bd')

plt.tight_layout()
plt.show()